# Simple Bat Chirp Detector (No ML)

This notebook implements a lightweight band‑pass energy detector for ultrasonic bat calls. It:
1. Loads the raw 384 kHz wav.
2. Applies a 20‑100 kHz Butterworth band‑pass.
3. Extracts the envelope via Hilbert transform + low‑pass.
4. Uses an adaptive threshold based on a running median of the envelope.
5. Finds contiguous supra‑threshold regions, merges short gaps, adds padding.
6. Writes each detection as a short wav clip (≈30‑50 ms) plus a JSON side‑car with timing info.

You can run the notebook as‑is or copy the functions into a Waggle plugin.


In [1]:
import numpy as np
import soundfile as sf
from scipy import signal
from pathlib import Path
import json
import os

# -------------------------- USER PARAMETERS --------------------------
LOWCUT   = 20_000      # Hz
HIGHCUT  = 100_000     # Hz
FORDER   = 4           # Butterworth order
ENV_LP_CUT = 500.0     # Hz, low‑pass for envelope
MED_WIN_SEC = 0.5      # seconds for running median of envelope
THRESH_FACTOR = 4.0    # multiplier over median
MIN_DUR_SEC = 0.001    # minimum length of a detection (s)
MAX_GAP_SEC = 0.005    # merge gap if shorter than this (s)
PAD_PRE_SEC = 0.010    # seconds of padding before onset
PAD_POST_SEC = 0.010   # seconds of padding after offset
OUTPUT_DIR = Path("detected_clips")
SAVE_AS_INT16 = True   # write PCM_16 wav (smaller, standard)
# -------------------------------------------------------------------


In [2]:
def design_bandpass(fs, low, high, order=4):
    nyq = 0.5 * fs
    low_n  = low / nyq
    high_n = high / nyq
    b, a = signal.butter(order, [low_n, high_n], btype='band')
    return b, a

def envelope(sig, fs, lowpass_hz):
    analytic = signal.hilbert(sig)
    amp = np.abs(analytic)
    nyq = 0.5 * fs
    wn = lowpass_hz / nyq
    b, a = signal.butter(2, wn, btype='low')
    env = signal.filtfilt(b, a, amp)
    return env

def detect_segments(env, fs, thresh, min_dur, max_gap):
    above = env > thresh
    diff = np.diff(astype(int, above))
    onsets = np.where(diff == 1)[0] + 1
    offsets = np.where(diff == -1)[0] + 1
    if above[0]:
        onsets = np.insert(onsets, 0, 0)
    if above[-1]:
        offsets = np.append(offsets, len(above))
    starts = onsets
    ends   = offsets
    min_samples = int(np.ceil(min_dur * fs))
    valid = (ends - starts) >= min_samples
    starts = starts[valid]
    ends   = ends[valid]
    max_gap_samples = int(np.ceil(max_gap * fs))
    merged_st = []
    merged_en = []
    s = starts[0]
    e = ends[0]
    for ns, ne in zip(starts[1:], ends[1:]):
        if ns - e <= max_gap_samples:
            e = ne
        else:
            merged_st.append(s)
            merged_en.append(e)
            s, e = ns, ne
    merged_st.append(s)
    merged_en.append(e)
    return list(zip(merged_st, merged_en))

def write_clip(wav_data, sr, start, end, out_path, prefix="", meta=None):
    clip = wav_data[start:end]
    if SAVE_AS_INT16:
        clip_int = np.int16(np.clip(clip, -1.0, 1.0) * 32767)
        sf.write(out_path, clip_int, sr, subtype='PCM_16')
    else:
        sf.write(out_path, clip, sr, subtype='FLOAT')
    if meta:
        json_path = out_path.with_suffix('.json')
        with json_path.open('w') as f:
            json.dump(meta, f, indent=2)

def main(wav_path):
    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    wav, sr = sf.read(wav_path, always_2d=False)
    if wav.ndim > 1:
        wav = np.mean(wav, axis=1)
    print(f'Loaded {Path(wav_path).name}: {len(wav)/sr:.2f} s @ {sr} Hz')
    
    b, a = design_bandpass(sr, LOWCUT, HIGHCUT, FORDER)
    filtered = signal.filtfilt(b, a, wav)
    
    env = envelope(filtered, sr, ENV_LP_CUT)
    
    win_len = int(MED_WIN_SEC * sr)
    if win_len % 2 == 0:
        win_len += 1
    med = signal.medfilt(env, kernel_size=win_len)
    thresh = med * THRESH_FACTOR
    
    segs = detect_segments(env, sr, thresh, MIN_DUR_SEC, MAX_GAP_SEC)
    print(f'Found {len(segs)} raw segments after thresholding.')
    
    pad_pre = int(PAD_PRE_SEC * sr)
    pad_post = int(PAD_POST_SEC * sr)
    saved = 0
    for i, (s, e) in enumerate(segs):
        s_pad = max(0, s - pad_pre)
        e_pad = min(len(wav), e + pad_post)
        t_start = s_pad / sr
        t_end   = e_pad / sr
        peak_amp = np.max(np.abs(wav[s_pad:e_pad]))
        meta = {
            "source_file": str(Path(wav_path).name),
            "start_sec": round(t_start, 6),
            "end_sec":   round(t_end, 6),
            "duration_s": round(t_end - t_start, 6),
            "peak_amplitude": float(peak_amp),
            "detection_index": i
        }
        out_name = f"{Path(wav_path).stem}_det{i:04d}.wav"
        out_path = out_dir / out_name
        write_clip(wav, sr, s_pad, e_pad, out_path, meta=meta)
        saved += 1
    print(f"Wrote {saved} clips to {out_dir}/")

# Run the detector on the supplied file
wav_file = "20260414_100939.wav"
main(wav_file)

# Optional: quick visualization of first 5 seconds
import matplotlib.pyplot as plt
wav, sr = sf.read(wav_file, always_2d=False)
if wav.ndim > 1:
    wav = np.mean(wav, axis=1)
b, a = design_bandpass(sr, LOWCUT, HIGHCUT, FORDER)
filtered = signal.filtfilt(b, a, wav)
env = envelope(filtered, sr, ENV_LP_CUT)
win_len = int(MED_WIN_SEC * sr)
if win_len % 2 == 0:
    win_len += 1
med = signal.medfilt(env, kernel_size=win_len)
thresh = med * THRESH_FACTOR

# limit to first 5 sec
max_samples = int(5 * sr)
t = np.arange(min(len(wav), max_samples)) / sr
plt.figure(figsize=(12,4))
plt.plot(t, wav[:len(t)], label='raw (zoomed)', alpha=0.4)
plt.plot(t, filtered[:len(t)], label='band‑passed', alpha=0.6)
plt.plot(t, env[:len(t)], label='envelope', linewidth=1.5)
plt.plot(t, thresh[:len(t)], label='threshold (adaptive)', linestyle='--')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('First 5 s: raw, filtered, envelope, threshold')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


Loaded 20260414_100939.wav: 600.00 s @ 384000 Hz


NameError: name 'astype' is not defined